# 10_03 — Color Histogram + Logistic Regression Baseline (Classical ML on Handcrafted Features)

## Goal
Train and benchmark a fast classical baseline using **HSV color histogram** features and **Logistic Regression** on the fixed dataset split **split_v1** created in Phase 1.

This notebook is designed to be:
- **Re-runnable** (auto-detects cached features; extracts only if missing)
- **Parallel-safe** (unique `RUN_ID`, no metric file overwrites)
- **Reproducible** (saved config + metrics + MLflow logging)

This notebook produces:
- A trained scikit-learn pipeline (`StandardScaler` → `LogisticRegression`)
- Metrics: accuracy, macro-F1, precision, recall, confusion matrix (+ full classification report)
- Serialized artifacts under:
  - `models/ml_basic_features/colorhist_lr/run_YYYYMMDD_HHMMSS/`
- A summary metrics file under:
  - `reports/metrics/colorhist_lr_logreg_metrics.json`
- MLflow logging (params, metrics, artifacts) to:
  - `mlruns/`

## Data Contract (Reusing Phase 1 Infrastructure)
- Splits are loaded from:
  - `data/splits/split_v1/train.csv`
  - `data/splits/split_v1/val.csv`
  - `data/splits/split_v1/test.csv`
- Class mapping is loaded from:
  - `data/splits/split_v1/classes.json`
- No dataset files are modified.

## Feature Pipeline (Cached)
We compute features from **pixel-space images**:
- Load image with PIL in RGB
- Resize to fixed size (224×224)
- Convert to HSV
- Compute histograms for H, S, V channels (default: 32 bins each)
- Concatenate → 96-dim feature vector
- L1-normalize the concatenated histogram for stability

Features are cached automatically to avoid recomputation:
- `data/processed/features/split_v1/colorhist_hsv32/`

If the cache exists, it is reused automatically.

## Verification
At the end, the notebook reloads the saved model and runs a small `predict` sanity check to confirm serialization and inference work.

In [25]:
# Cell 1 — Imports

import os
import json
import pickle
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

import mlflow

In [26]:
# Cell 2 — Project root + paths + run metadata (parallel-safe)

NOTEBOOK_DIR = Path.cwd()

candidates = [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents)
PROJECT_ROOT = None
for p in candidates:
    if (p / "data").exists() and (p / "src").exists() and (p / "configs").exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f"Could not locate project root from: {NOTEBOOK_DIR}")

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

SPLIT_ID = "split_v1"
SPLITS_DIR = DATA_DIR / "splits" / SPLIT_ID

TRAIN_CSV = SPLITS_DIR / "train.csv"
VAL_CSV = SPLITS_DIR / "val.csv"
TEST_CSV = SPLITS_DIR / "test.csv"
CLASSES_JSON = SPLITS_DIR / "classes.json"

MODEL_NAME = "colorhist_lr"
FEATURE_TYPE = "colorhist_hsv"
CLASSIFIER_NAME = "logreg"
TRANSFORM_ID = "pixelspace_resize224_hsv_hist32x3_l1norm"

# Unique per run; safe for running several notebooks at once
RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")

RUN_DIR = MODELS_DIR / "ml_basic_features" / MODEL_NAME / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Keep report filename stable (plan expects colorhist_lr_metrics.json)
# and ALSO create a classifier-specific file to avoid overwrites when running variants.
REPORT_METRICS_PATH = REPORTS_METRICS_DIR / f"{MODEL_NAME}_metrics.json"
REPORT_METRICS_PATH_VARIANT = REPORTS_METRICS_DIR / f"{MODEL_NAME}_{CLASSIFIER_NAME}_metrics.json"

# Feature cache location
FEATURES_DIR = DATA_DIR / "processed" / "features" / SPLIT_ID / "colorhist_hsv32"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)
print("FEATURES_DIR:", FEATURES_DIR)

PROJECT_ROOT: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server
RUN_DIR: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/ml_basic_features/colorhist_lr/run_20260311_194949
FEATURES_DIR: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/data/processed/features/split_v1/colorhist_hsv32


In [27]:
# Cell 3 — MLflow setup (project-root tracking dir)

TRACKING_DIR = PROJECT_ROOT / "mlruns"
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

print("MLflow tracking URI:", TRACKING_DIR.as_uri())

MLflow tracking URI: file:///home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/mlruns


In [28]:
# Cell 4 — Load split CSVs + classes mapping

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    class_to_idx = json.load(f)

idx_to_class = {v: k for k, v in class_to_idx.items()}

train_df["label_idx"] = train_df["label"].map(class_to_idx)
val_df["label_idx"] = val_df["label"].map(class_to_idx)
test_df["label_idx"] = test_df["label"].map(class_to_idx)

assert train_df["label_idx"].isna().sum() == 0
assert val_df["label_idx"].isna().sum() == 0
assert test_df["label_idx"].isna().sum() == 0

print("Train/Val/Test sizes:", len(train_df), len(val_df), len(test_df))
train_df.head()

Train/Val/Test sizes: 50127 6266 6266


,filepath,label,label_idx
0,f:/Projects/AnimalClassification/data/prepared...,wildlife,2
1,f:/Projects/AnimalClassification/data/prepared...,dogs,1
2,f:/Projects/AnimalClassification/data/prepared...,dogs,1
3,f:/Projects/AnimalClassification/data/prepared...,wildlife,2
4,f:/Projects/AnimalClassification/data/prepared...,cats,0


In [29]:
# Cell 5 — Feature configuration (HSV color histograms)

IMG_SIZE = (224, 224)

H_BINS = 32
S_BINS = 32
V_BINS = 32

# Histogram ranges for HSV in PIL:
# H channel typically in [0, 255], same for S and V after conversion to HSV in PIL.
H_RANGE = (0, 256)
S_RANGE = (0, 256)
V_RANGE = (0, 256)

FEATURE_DIM = H_BINS + S_BINS + V_BINS  # expected: 96

print("IMG_SIZE:", IMG_SIZE)
print("Bins:", (H_BINS, S_BINS, V_BINS), "FEATURE_DIM:", FEATURE_DIM)

IMG_SIZE: (224, 224)
Bins: (32, 32, 32) FEATURE_DIM: 96


In [30]:
# Cell 6 — Feature extraction helpers

def load_image_resize_rgb(path: str) -> Image.Image:
    img = Image.open(path).convert("RGB")
    img = img.resize(IMG_SIZE)
    return img

def colorhist_features_from_path(path: str) -> np.ndarray:
    """
    Returns a 96-dim HSV histogram feature vector (float32).
    Steps:
      - RGB -> HSV (PIL)
      - histogram for H,S,V (32 bins each)
      - concatenate + L1 normalize
    """
    img = load_image_resize_rgb(path)
    hsv = img.convert("HSV")
    hsv_arr = np.array(hsv, dtype=np.uint8)  # shape: (H,W,3)

    h = hsv_arr[:, :, 0].ravel()
    s = hsv_arr[:, :, 1].ravel()
    v = hsv_arr[:, :, 2].ravel()

    h_hist, _ = np.histogram(h, bins=H_BINS, range=H_RANGE)
    s_hist, _ = np.histogram(s, bins=S_BINS, range=S_RANGE)
    v_hist, _ = np.histogram(v, bins=V_BINS, range=V_RANGE)

    feat = np.concatenate([h_hist, s_hist, v_hist]).astype(np.float32)

    # L1 normalize (avoid division by zero)
    denom = feat.sum()
    if denom > 0:
        feat = feat / denom

    return feat

In [31]:
# Cell 7 — Legacy full extraction code (kept commented for documentation / reproducibility)

# def build_features(df: pd.DataFrame):
#     X = np.vstack([colorhist_features_from_path(p) for p in df["filepath"].values])
#     y = df["label_idx"].values.astype(np.int64)
#     return X, y
#
# X_train, y_train = build_features(train_df)
# X_val, y_val = build_features(val_df)
# X_test, y_test = build_features(test_df)
#
# X_train.shape, X_val.shape, X_test.shape

In [32]:
# Cell 8 — Feature cache auto-detect + load/extract

def cache_paths(base_dir: Path):
    return {
        "train": base_dir / "train.npy",
        "val": base_dir / "val.npy",
        "test": base_dir / "test.npy",
        "labels_train": base_dir / "labels_train.npy",
        "labels_val": base_dir / "labels_val.npy",
        "labels_test": base_dir / "labels_test.npy",
    }

paths = cache_paths(FEATURES_DIR)

cache_exists = all(p.exists() for p in paths.values())

if cache_exists:
    print(f"[CACHE] Found cached features at: {FEATURES_DIR}")
    X_train = np.load(paths["train"], mmap_mode="r")
    X_val = np.load(paths["val"], mmap_mode="r")
    X_test = np.load(paths["test"], mmap_mode="r")

    y_train = np.load(paths["labels_train"])
    y_val = np.load(paths["labels_val"])
    y_test = np.load(paths["labels_test"])
else:
    print(f"[CACHE MISS] Extracting color histogram features and saving to: {FEATURES_DIR}")

    def build_features(df: pd.DataFrame):
        X_list = []
        for p in df["filepath"].values:
            # Fix Windows absolute paths for Ubuntu compatibility
            path_str = str(p).replace("\\", "/")
            if "data/prepared" in path_str:
                rel_path = path_str.split("data/prepared/")[1]
                true_path = PROJECT_ROOT / "data" / "prepared" / rel_path
            else:
                true_path = Path(p)
            
            X_list.append(colorhist_features_from_path(str(true_path)))
            
        X = np.vstack(X_list)
        y = df["label_idx"].values.astype(np.int64)
        return X, y

    X_train, y_train = build_features(train_df)
    X_val, y_val = build_features(val_df)
    X_test, y_test = build_features(test_df)

    # Save features
    np.save(paths["train"], X_train.astype(np.float32))
    np.save(paths["val"], X_val.astype(np.float32))
    np.save(paths["test"], X_test.astype(np.float32))

    np.save(paths["labels_train"], y_train)
    np.save(paths["labels_val"], y_val)
    np.save(paths["labels_test"], y_test)

    print("[CACHE] Saved feature arrays.")

print("X_train:", X_train.shape, X_train.dtype)
print("X_val:", X_val.shape, X_val.dtype)
print("X_test:", X_test.shape, X_test.dtype)
print("y_train:", y_train.shape, y_train.dtype, "unique:", np.unique(y_train))

[CACHE MISS] Extracting color histogram features and saving to: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/data/processed/features/split_v1/colorhist_hsv32


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[CACHE] Saved feature arrays.
X_train: (50127, 96) float32
X_val: (6266, 96) float32
X_test: (6266, 96) float32
y_train: (50127,) int64 unique: [0 1 2]


In [33]:
# Cell 9 — Sanity checks

assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]
assert X_test.shape[0] == y_test.shape[0]

assert X_train.shape[1] == FEATURE_DIM
assert X_val.shape[1] == FEATURE_DIM
assert X_test.shape[1] == FEATURE_DIM

# fast finite check on a slice
assert np.isfinite(np.asarray(X_train[:1000])).all()
assert set(np.unique(y_train)).issubset(set(class_to_idx.values()))

print("[OK] Sanity checks passed.")

[OK] Sanity checks passed.


In [34]:
# Cell 10 — Train Logistic Regression (fast and scalable)

LOGREG_PARAMS = {
    "solver": "saga",
    "C": 2.0,
    "max_iter": 500,
    "random_state": 42,
    "n_jobs": -1,
}

model = Pipeline([
    ("scaler", StandardScaler(with_mean=False, with_std=True)),
    ("clf", LogisticRegression(**LOGREG_PARAMS)),
])

model.fit(X_train, y_train)
print("[OK] Training complete.")

/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


[OK] Training complete.


In [35]:
# Cell 11 — Evaluation

def evaluate(model, X, y):
    preds = model.predict(X)
    metrics = {
        "accuracy": float(accuracy_score(y, preds)),
        "f1_macro": float(f1_score(y, preds, average="macro")),
        "precision_macro": float(precision_score(y, preds, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y, preds, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(y, preds).tolist(),
        "classification_report": classification_report(
            y,
            preds,
            target_names=[idx_to_class[i] for i in sorted(idx_to_class.keys())],
            zero_division=0,
            output_dict=True,
        ),
    }
    return metrics

val_metrics = evaluate(model, X_val, y_val)
test_metrics = evaluate(model, X_test, y_test)

print("VAL accuracy:", val_metrics["accuracy"], "VAL f1_macro:", val_metrics["f1_macro"])
print("TEST accuracy:", test_metrics["accuracy"], "TEST f1_macro:", test_metrics["f1_macro"])

VAL accuracy: 0.509256303862113 VAL f1_macro: 0.5103794852004446
TEST accuracy: 0.511490584104692 TEST f1_macro: 0.5123131147473816


In [36]:
# Cell 12 — Save artifacts (per-run + reports)

config = {
    "model_name": MODEL_NAME,
    "feature_type": FEATURE_TYPE,
    "classifier": CLASSIFIER_NAME,
    "split_id": SPLIT_ID,
    "transform_id": TRANSFORM_ID,
    "img_size": list(IMG_SIZE),
    "run_id": RUN_ID,
    "feature_params": {
        "h_bins": H_BINS,
        "s_bins": S_BINS,
        "v_bins": V_BINS,
        "h_range": list(H_RANGE),
        "s_range": list(S_RANGE),
        "v_range": list(V_RANGE),
        "feature_dim": FEATURE_DIM,
        "normalization": "l1",
    },
    "logreg_params": LOGREG_PARAMS,
    "feature_cache_dir": str(FEATURES_DIR),
}

metrics_payload = {
    "val": val_metrics,
    "test": test_metrics,
}

model_path = RUN_DIR / "model.pkl"
config_path = RUN_DIR / "config.json"
metrics_path = RUN_DIR / "metrics.json"

with open(model_path, "wb") as f:
    pickle.dump(model, f)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

# Plan-expected name (may be overwritten by subsequent reruns—acceptable per plan)
with open(REPORT_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

# Variant-safe name (won't overwrite)
with open(REPORT_METRICS_PATH_VARIANT, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

print("Saved:")
print(" -", model_path)
print(" -", config_path)
print(" -", metrics_path)
print(" -", REPORT_METRICS_PATH)
print(" -", REPORT_METRICS_PATH_VARIANT)

Saved:
 - /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/ml_basic_features/colorhist_lr/run_20260311_194949/model.pkl
 - /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/ml_basic_features/colorhist_lr/run_20260311_194949/config.json
 - /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/models/ml_basic_features/colorhist_lr/run_20260311_194949/metrics.json
 - /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/reports/metrics/colorhist_lr_metrics.json
 - /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/reports/metrics/colorhist_lr_logreg_metrics.json


In [37]:
# Cell 13 — MLflow logging

with mlflow.start_run(run_name=f"{MODEL_NAME}_{RUN_ID}"):
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("feature_type", FEATURE_TYPE)
    mlflow.log_param("classifier", CLASSIFIER_NAME)
    mlflow.log_param("split_id", SPLIT_ID)
    mlflow.log_param("transform_id", TRANSFORM_ID)
    mlflow.log_param("run_id", RUN_ID)

    # feature params
    mlflow.log_param("img_size", str(IMG_SIZE))
    mlflow.log_param("h_bins", H_BINS)
    mlflow.log_param("s_bins", S_BINS)
    mlflow.log_param("v_bins", V_BINS)
    mlflow.log_param("normalization", "l1")

    # logreg params
    for k, v in LOGREG_PARAMS.items():
        mlflow.log_param(f"logreg_{k}", v)

    # metrics
    mlflow.log_metric("val_accuracy", val_metrics["accuracy"])
    mlflow.log_metric("val_f1_macro", val_metrics["f1_macro"])
    mlflow.log_metric("test_accuracy", test_metrics["accuracy"])
    mlflow.log_metric("test_f1_macro", test_metrics["f1_macro"])

    # artifacts
    mlflow.log_artifact(str(model_path))
    mlflow.log_artifact(str(config_path))
    mlflow.log_artifact(str(metrics_path))

print("[OK] MLflow run logged.")

[OK] MLflow run logged.


In [38]:
# Cell 14 — Verification: reload model + inference sanity check

with open(model_path, "rb") as f:
    loaded_model = pickle.load(f)

sample_X = np.asarray(X_test[:8])  # ensure concrete array for mmap
sample_preds = loaded_model.predict(sample_X)
sample_pred_labels = [idx_to_class[int(i)] for i in sample_preds]

print("Sample predictions:", sample_pred_labels)

Sample predictions: ['cats', 'dogs', 'wildlife', 'cats', 'dogs', 'cats', 'dogs', 'wildlife']
